# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-08','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** one line item from one order

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total
vendors = pd.json_normalize(vendors_json)
lines = pd.json_normalize(orders_json,record_path='lines',meta=['order', 'vendor_id'],sep='.',errors='ignore')
assert len(lines) == 4
assert sum(lines['qty']) == 7

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
# TODO
vendors = pd.DataFrame(vendors_json)
joined = lines.merge(vendors, on = 'vendor_id', how = 'left', indicator=True)
unmatched = joined[joined['_merge'] == 'left_only']
print(f"Unmatched vendor is {unmatched['vendor_id'].iloc[0]}, units are {unmatched['qty'].sum()}")
joined = joined.drop(columns=['_merge'])
joined

Unmatched vendor is V-08, units are 3


,item,qty,order,vendor_id,name,zone
0,Cheeseburger,2,1,V-01,Hoos Burgers,A
1,Soda,1,1,V-01,Hoos Burgers,A
2,Foam Finger,1,2,V-10,Cav Merch,A
3,Hot Dog,3,3,V-08,NaN,NaN


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# TODO
joined['name'] = joined['name'].fillna('Unknown')

q3 = joined.groupby('name')['qty'].sum()
print("The unmatched vendor appears as 'Unknown' because it has no matching vendor name, but we still want to keep it so we're not dropping 3 items from the total qty")
q3

The unmatched vendor appears as 'Unknown' because it has no matching vendor name, but we still want to keep it so we're not dropping 3 items from the total qty


name
Cav Merch       1
Hoos Burgers    3
Unknown         3
Name: qty, dtype: int64

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# TODO
q4 = joined.groupby('zone')['qty'].sum()
print(f"The total items sold is {q4.sum()}, and Zone A sold the most items. The difference is {7 - q4.sum()} and those units belong to V-08")
q4

The total items sold is 4, and Zone A sold the most items. The difference is 3 and those units belong to V-08


zone
A    4
Name: qty, dtype: int64

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]
for order in ragged_json:
    order.setdefault('lines', [])

ragged = pd.json_normalize(ragged_json,record_path='lines',meta=['order', 'vendor_id'],errors='ignore')
ragged
# TODO

,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,Fries,NaN,6,V-01


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?



```
# This is formatted as code
```

a) One row per line item is the useful shape because it allows us to perform item-specific operations or queries like the total sold. With one row per order, the lines themselves get nested which would end up requiring some sort of looping to get any item-specific answers, and that can get messy so doing one row per line item is easier in that regard

b) The practical difference between a missing quantity and a quantity of zero is what they represent. A missing quantity would be one that's unknown, whereas a quantity of zero just means there was a quantity of zero for that item. If we conflate the two, incomplete data would be treated as valid, which would affect metrics like average or total.